# Multi-Site / Multi-Identifier Rename Benchmark

Validate how well small open-source code LLMs (**DeepSeek-Coder**, **Qwen2.5-Coder**) recover **multiple identifiers, each at all of its usage sites**, in real Java files from famous OSS repos.

**Runtime → Change runtime type → GPU.** T4 is enough for the 0.5–1.5B models; pick A100 for 6.7–7B.

Run the cells top to bottom.

## 1. Get the code

In [ ]:
# Clone the repo (or skip this cell if you mounted Google Drive)
import os
if not os.path.isdir('CoRefusion'):
    !git clone https://github.com/ANONYMOUS/CoReFusion
%cd CoRefusion/experiments/multi_site_rename

In [ ]:
!pip -q install -r requirements.txt

## 2. Build the dataset from famous repos

Fetches the Java files listed in `repos.json`, masks `--num-identifiers` identifiers per instance (each at all of its sites), and writes `data/multisite_rename.jsonl`. Files that fail to fetch/parse are skipped; the bundled `data/sample_java/` file is always included as a fallback.

Add `--strip-comments` for a stricter setting that prevents comments from leaking the original names.

In [ ]:
!python build_dataset.py \
    --num-identifiers 3 --min-sites 2 --instances-per-file 2 --max-chars 9000

# peek at one instance
import json
rows = [json.loads(l) for l in open('data/multisite_rename.jsonl')]
print(f'{len(rows)} instances')
print('example ground truth:', rows[0]['ground_truth'])
print('example sites       :', rows[0]['sites'])

## 3. Run the benchmark

Edit `--models` to taste (keys are defined in `models.py`). Drop `--limit` for the full set.

In [ ]:
!python run_benchmark.py \
    --models qwen2.5-1.5b-instruct deepseek-1.3b-instruct \
    --out-dir results --limit 100

## 4. Inspect results

In [ ]:
import pandas as pd
df = pd.read_csv('results/summary_all.csv')
display(df)

# a few qualitative examples from the first model
import glob, json
pred_file = sorted(glob.glob('results/*_predictions.jsonl'))[0]
for line in list(open(pred_file))[:3]:
    r = json.loads(line)
    print('=' * 60)
    print(r['id'], '| EM =', round(r['score']['em'], 2))
    for ph, info in r['per_identifier'].items():
        flag = 'OK ' if info['em'] else '   '
        print(f"  {flag}{ph}: gold={info['gold']!r:24} pred={info['pred']!r}")